# 02 · Feature Engineering

**Input:** `silver_results.parquet`  
**Output:** `gold_train.parquet`, `gold_test.parquet`  
**Split:** chronological — train ≤ 2019-12-31 / test ≥ 2020-01-01  
**Leakage prevention:** `.shift(1)` before every rolling aggregation

| Group | Features |
|-------|----------|
| Form | `rolling_form_3/5/10`, `rolling_margin_3` |
| Head-to-head | `h2h_winrate` |
| Momentum | `consecutive_wins` |
| Recovery | `days_since_prev` |
| Experience | `experience`, `experience_diff` |
| RWC pedigree | `rwc_appearances`, `rwc_best_stage`, `rwc_cumul_score` |
| Strength | `elo_diff_pre` |

In [1]:
# Setup
from pathlib import Path
import pandas as pd
import numpy as np

SILVER_PATH           = Path('../data/silver/silver_results.parquet')
GOLD_DIR              = Path('../data/gold')
GOLD_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_END, TEST_START = '2019-12-31', '2020-01-01'
ELO_INIT, ELO_K       = 1500, 32

In [2]:
df = pd.read_parquet(SILVER_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date','team']).reset_index(drop=True)
print(f'Loaded: {len(df):,} | {df.team.nunique()} teams | '
      f'{df.date.min().date()} – {df.date.max().date()}')

Loaded: 3,410 | 28 teams | 1987-05-22 – 2024-08-17


In [3]:
# shift(1) = only past matches — no leakage
parts = []
for _, g in df.groupby('team', sort=False):
    g = g.sort_values('date').copy()
    s, m = g.win.shift(1), g.score_margin.shift(1)
    g['rolling_form_3']   = s.rolling(3,  min_periods=1).mean()
    g['rolling_form_5']   = s.rolling(5,  min_periods=1).mean()
    g['rolling_form_10']  = s.rolling(10, min_periods=3).mean()
    g['rolling_margin_3'] = m.rolling(3,  min_periods=1).mean()
    parts.append(g)
df = pd.concat(parts).sort_values(['date','team']).reset_index(drop=True)
print('Rolling features done')

Rolling features done


In [4]:
# consecutive wins — shift(1) excludes current match
df = df.sort_values(['team','date']).reset_index(drop=True)
for team, idx in df.groupby('team').groups.items():
    shifted = df.loc[idx, 'win'].shift(1).fillna(0).astype(int).values
    streak, cnt = [], 0
    for w in shifted:
        cnt = cnt + 1 if w else 0
        streak.append(cnt)
    df.loc[idx, 'consecutive_wins'] = streak
print(f'Consecutive wins done | max streak: {df.consecutive_wins.max():.0f}')

Consecutive wins done | max streak: 18


In [5]:
# ── head-to-head win rate — O(n log n) via groupby ──────────────────────
# For each (team, opponent) pair, compute cumulative win rate
# using shift(1) to exclude current match (no leakage).

df = df.sort_values(['team','opponent','date']).reset_index(drop=True)
grp = df.groupby(['team','opponent'])
df['h2h_wins']    = grp['win'].transform(lambda x: x.shift(1).expanding().sum())
df['h2h_matches']  = grp['win'].transform(lambda x: x.shift(1).expanding().count())
df['h2h_winrate']  = (df.h2h_wins / df.h2h_matches).fillna(0.5)
df.drop(columns=['h2h_wins','h2h_matches'], inplace=True)
df = df.sort_values(['date','team']).reset_index(drop=True)
print(f'H2H done (O(n log n)) | mean: {df.h2h_winrate.mean():.3f}')


H2H done (O(n log n)) | mean: 0.493


In [6]:
df = df.sort_values(['team','date']).reset_index(drop=True)
df['days_since_prev'] = df.groupby('team')['date'].diff().dt.days.fillna(30)

In [7]:
# cumulative caps — no leakage via cumcount()
df = df.sort_values(['team','date']).reset_index(drop=True)
df['experience'] = df.groupby('team').cumcount()
exp_map               = df.set_index(['team','date','opponent'])['experience']
df['exp_opp']         = [exp_map.get((r.opponent,r.date,r.team),0) for _,r in df.iterrows()]
df['experience_diff'] = df.experience - df.exp_opp
print(f'Experience done | max caps: {df.experience.max()}')

Experience done | max caps: 353


In [8]:
# ── tournament tier ─────────────────────────────────────────────────────
# Ordinal encoding based on competitive importance and qualification difficulty.
# Adapted from Gásquez & Royuela (2016) who use competition prestige
# as a contextual feature in football outcome prediction.
#   3 = Rugby World Cup (global, highest prestige)
#   2 = Six Nations / Rugby Championship / Lions (continental elite)
#   1 = Autumn Nations / Pacific Nations (tier 1 invite tournaments)
#   0 = Other (test matches, mid-year tours)

TIERS = {
    3: ['rugby world cup','world cup'],
    2: ['six nations','rugby championship','tri nations',
        'british & irish lions','british and irish lions'],
    1: ['autumn nations','summer nations','autumn internationals',
        'summer internationals','pacific nations'],
}
def get_tier(name):
    n = str(name).lower()
    for tier, kws in TIERS.items():
        if any(k in n for k in kws): return tier
    return 0

df['tournament_tier'] = df.tournament.apply(get_tier).astype('int8')
print(f'Tournament tiers: {df.tournament_tier.value_counts().sort_index().to_dict()}')

Tournament tiers: {0: 1170, 1: 44, 2: 1214, 3: 982}


In [9]:
# ── Elo rating — computed on match level (not row level) ─────────────
# IMPORTANT: silver has 2 rows per match (home + away perspective).
# Elo must be updated once per match, not twice.
# Solution: compute Elo on deduplicated match level, then join back.

def calc_elo_matches(df_in, k=ELO_K, init=ELO_INIT):
    """Compute Elo on unique matches (one update per match)."""
    # get unique matches: home perspective only
    matches = (df_in[df_in.home == 1]
               .sort_values('date')[['date','team','opponent','win']]
               .copy())
    elos = {t: init for t in
            set(matches.team) | set(matches.opponent)}
    records = []
    for _, r in matches.iterrows():
        e_t = elos[r.team]
        e_o = elos.get(r.opponent, init)
        exp = 1 / (1 + 10**((e_o - e_t) / 400))
        records.append({
            'date': r.date, 'team': r.team,
            'opponent': r.opponent,
            'elo_pre_home': e_t,
            'elo_pre_away': e_o,
        })
        # update ONCE per match
        elos[r.team]     = e_t + k * (r.win - exp)
        elos[r.opponent] = e_o + k * ((1 - r.win) - (1 - exp))
    return pd.DataFrame(records)

elo_df = calc_elo_matches(df)

# join elo back to both perspectives
df = df.merge(
    elo_df[['date','team','opponent','elo_pre_home']]
          .rename(columns={'elo_pre_home': 'elo_pre',
                             'team': 'team', 'opponent': 'opponent'}),
    on=['date','team','opponent'], how='left'
)
df = df.merge(
    elo_df[['date','team','opponent','elo_pre_away']]
          .rename(columns={'elo_pre_away': 'elo_pre',
                             'team': 'opponent', 'opponent': 'team'}),
    on=['date','team','opponent'], how='left', suffixes=('','_away')
)
# combine: home perspective has elo_pre, away has elo_pre_away
df['elo_pre'] = df['elo_pre'].fillna(df.get('elo_pre_away', float('nan')))
if 'elo_pre_away' in df.columns:
    df.drop(columns=['elo_pre_away'], inplace=True)
df['elo_pre'] = df['elo_pre'].fillna(ELO_INIT)

# elo_opp = elo_pre of opponent in same match
elo_map = df.set_index(['date','team','opponent'])['elo_pre']
df['elo_opp'] = df.apply(
    lambda r: elo_map.get((r.date, r.opponent, r.team), ELO_INIT), axis=1
)
df['elo_diff_pre'] = df.elo_pre - df.elo_opp
print(f'Elo done (match-level, no double update) | '
      f'range: {df.elo_pre.min():.0f}–{df.elo_pre.max():.0f}')


Elo done (match-level, no double update) | range: 1203–2096


In [10]:
FILL_05 = ['rolling_form_3','rolling_form_5','rolling_form_10','h2h_winrate']
FILL_0  = ['rolling_margin_3','consecutive_wins','experience',
           'exp_opp','experience_diff']
df[FILL_05] = df[FILL_05].fillna(0.5)
df[FILL_0]  = df[FILL_0].fillna(0.0)
df['tournament_tier'] = df.tournament_tier.fillna(0).astype('int8')
assert df[FILL_05+FILL_0].isna().sum().sum() == 0
print('fillna done | 0 NaN confirmed')

fillna done | 0 NaN confirmed


In [11]:
df_train = df[df.date <= TRAIN_END].copy()
df_test  = df[df.date >= TEST_START].copy()
print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')
print(f'Win rate — train: {df_train.win.mean():.1%} | test: {df_test.win.mean():.1%}')
print(f'Teams  — train: {df_train.team.nunique()} | test: {df_test.team.nunique()}')

Train: 2,918 | Test: 492
Win rate — train: 49.1% | test: 48.4%
Teams  — train: 25 | test: 23


In [12]:
df_train.to_parquet(GOLD_DIR / 'gold_train.parquet', index=False)
df_test.to_parquet( GOLD_DIR / 'gold_test.parquet',  index=False)
print(f'Saved: gold_train ({len(df_train):,}) | gold_test ({len(df_test):,})')

feat_cols = ['rolling_form_3','rolling_form_5','rolling_form_10','rolling_margin_3',
             'h2h_winrate','consecutive_wins','days_since_prev','experience',
             'experience_diff','tournament_tier',
             'rwc_appearances','rwc_best_stage','rwc_cumul_score','elo_diff_pre']
print('\nFeature summary (train):')
print(df_train[[c for c in feat_cols if c in df_train.columns]].describe().round(3).to_string())

Saved: gold_train (2,918) | gold_test (492)

Feature summary (train):
       rolling_form_3  rolling_form_5  rolling_form_10  rolling_margin_3  h2h_winrate  consecutive_wins  days_since_prev  experience  experience_diff  tournament_tier  rwc_appearances  rwc_best_stage  rwc_cumul_score  elo_diff_pre
count        2918.000        2918.000         2918.000          2918.000     2918.000          2918.000         2918.000    2918.000         2918.000         2918.000         2918.000        2918.000         2918.000      2918.000
mean            0.494           0.493            0.497             0.151        0.493             1.517           75.745     120.645            0.000            1.561            4.096           3.206           10.130         0.000
std             0.346           0.302            0.253            18.179        0.315             2.581          269.075      83.498           56.597            1.248            2.066           1.613            6.528       239.963
min   